# Hair Shifter — video inference on Colab

Re-renders a video with the hairstyle from a reference photo, using `src/inference.py`.

**Runtime → Change runtime type → GPU** before running. Run the cells top to bottom.

- The checkpoint comes from **Hugging Face** (default: the latest model) or from a **file path** (e.g. Google Drive).
- The default inputs are the samples in `assets/test_images/` (`phuc.mp4` + `ken.png`).

The project needs Python 3.10 (`pyproject.toml`), and Colab runs a newer Python. So the model runs in a 3.10 Poetry venv, which this notebook starts as a subprocess.

In [ ]:
#@title 1. Clone the repo { display-mode: "form" }
REPO_URL = "https://github.com/KatoVu/DP-AR-Hair.git"  #@param {type:"string"}
BRANCH = "feat/stability"  #@param {type:"string"}
REPO_DIR = "/content/DP-AR-Hair"

import os
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
#@title 2. Install tools (ninja, uv, poetry) { display-mode: "form" }
!command -v ninja >/dev/null 2>&1 || ( \
  wget -q https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip -O /tmp/ninja.zip && \
  sudo unzip -o /tmp/ninja.zip -d /usr/local/bin/ && \
  sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force )
!pip install -q uv poetry huggingface_hub

In [ ]:
#@title 3. Build the Python 3.10 environment (takes a while) { display-mode: "form" }
!uv python install 3.10
!poetry config virtualenvs.in-project true
!poetry env use "$(uv python find 3.10)"
VENV_PYTHON = f"{REPO_DIR}/.venv/bin/python"

# CLIP's setup.py imports pkg_resources, which new setuptools removed, so
# poetry's isolated build fails. Pre-install the locked commit against old
# setuptools; poetry then skips it as "Already installed".
import re
CLIP_REF = re.search(r'name = "clip".*?resolved_reference = "(\w+)"',
                     open("poetry.lock").read(), re.S).group(1)
!{VENV_PYTHON} -m ensurepip -q || true
!{VENV_PYTHON} -m pip install -q "setuptools<81" wheel
!{VENV_PYTHON} -m pip install -q --no-deps --no-build-isolation "git+https://github.com/openai/CLIP.git@{CLIP_REF}"

!poetry install --with vision,full --no-interaction

!{VENV_PYTHON} -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
#@title 4. Get the checkpoint { display-mode: "form" }
#@markdown **huggingface**: downloads `HF_FILENAME` from `HF_REPO`. For a private repo, add an `HF_TOKEN` secret (key icon in the left sidebar).
#@markdown **path**: uses `CHECKPOINT_PATH` as is. Google Drive gets mounted if the path is under `/content/drive`.
CHECKPOINT_SOURCE = "huggingface"  #@param ["huggingface", "path"]
HF_REPO = "jakekato/HairShifter-V1"  #@param {type:"string"}
HF_FILENAME = "epoch_0024_3335.pt"  #@param {type:"string"}
CHECKPOINT_PATH = "/content/drive/MyDrive/checkpoints/epoch_0100.pt"  #@param {type:"string"}

import os
import zipfile

if CHECKPOINT_SOURCE == "huggingface":
    from huggingface_hub import hf_hub_download
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None  # public repo, or no secret set
    CHECKPOINT = hf_hub_download(repo_id=HF_REPO, filename=HF_FILENAME, token=token)
else:
    if CHECKPOINT_PATH.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    CHECKPOINT = CHECKPOINT_PATH

if not os.path.isfile(CHECKPOINT):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")
# torch.save checkpoints are zip archives; anything else is a partial
# download, an unsynced Drive file or a Git LFS pointer
if not zipfile.is_zipfile(CHECKPOINT):
    raise RuntimeError(
        f"{CHECKPOINT} ({os.path.getsize(CHECKPOINT)} bytes) is not a valid PyTorch "
        "checkpoint. Re-download it, or wait for Drive to finish syncing.")
print(f"Checkpoint OK: {CHECKPOINT} ({os.path.getsize(CHECKPOINT) / 1e6:.1f} MB)")

In [ ]:
#@title 5. Run inference { display-mode: "form" }
VIDEO = "/content/DP-AR-Hair/assets/test_images/phuc.mp4"  #@param {type:"string"}
REFERENCE = "/content/DP-AR-Hair/assets/test_images/ken.png"  #@param {type:"string"}
OUTPUT = "/content/DP-AR-Hair/outputs/result.avi"  #@param {type:"string"}
BATCH_SIZE = 2  #@param {type:"integer"}
EMA = False  #@param {type:"boolean"}
ALIGN = False  #@param {type:"boolean"}
POOR = True  #@param {type:"boolean"}

import os
import subprocess
from IPython.display import Video

for label, path in [("Video", VIDEO), ("Reference", REFERENCE)]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{label} not found: {path}")
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

cmd = [f"{REPO_DIR}/.venv/bin/python", "src/inference.py",
       "--video", VIDEO, "--reference", REFERENCE,
       "--checkpoint", CHECKPOINT, "--output", OUTPUT,
       "--batch_size", str(BATCH_SIZE)]
cmd += [flag for flag, on in [("--ema", EMA), ("--align", ALIGN), ("--poor", POOR)] if on]

# Colab sets MPLBACKEND to its inline backend, which the 3.10 venv lacks
env = {**os.environ, "PYTHONPATH": "src:libs", "MPLBACKEND": "Agg"}

# Stream the log, and stop here if inference fails
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(f"Inference failed (exit code {proc.returncode}); see the log above.")

# MJPG .avi does not play in the browser; re-encode to H.264 for the preview
preview = os.path.splitext(OUTPUT)[0] + ".mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT,
                "-c:v", "libx264", "-pix_fmt", "yuv420p", preview], check=True)
print(f"Saved: {OUTPUT}\nPreview: {preview}")
Video(preview, embed=True, width=512)